<a href="https://colab.research.google.com/github/Shubham4521/Cost-Aware-LLM-Router/blob/main/notebooks/01_build_eval_set.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q datasets pandas

In [ ]:
import random, json
import pandas as pd
from datasets import load_dataset

random.seed(42)  # makes the random picks the same every time you run
rows = []
counters = {}

def add(category, difficulty, prompt, reference, answer_type,
        needs_search=False, extra=""):
    counters[category] = counters.get(category, 0) + 1
    rows.append({
        "id": f"{category}_{counters[category]:03d}",
        "category": category,
        "difficulty": difficulty,
        "needs_search": needs_search,
        "answer_type": answer_type,
        "prompt": prompt,
        "reference": reference,
        "extra": extra,
    })

def mc_prompt(question, labels, texts):
    options = "\n".join(f"{l}. {t}" for l, t in zip(labels, texts))
    return (f"{question}\n\n{options}\n\n"
            "Answer with only the letter of the correct option.")

print("Setup done")

Setup done


In [ ]:
math_ds = load_dataset("HuggingFaceH4/MATH-500", split="test")
MATH_INSTR = ("\n\nSolve the problem. On the last line write "
              "ANSWER: followed by only the final answer.")

for levels, diff in [([1, 2], "easy"), ([3], "medium"), ([4, 5], "hard")]:
    pool = [x for x in math_ds if x["level"] in levels]
    for x in random.sample(pool, 10):
        add("math", diff, x["problem"] + MATH_INSTR, x["answer"], "math_answer")

print("Math added:", counters["math"])

README.md:   0%|          | 0.00/412 [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/447k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/500 [00:00<?, ? examples/s]

Math added: 30


In [ ]:
mmlu = load_dataset("cais/mmlu", "all", split="test")

for i in random.sample(range(len(mmlu)), 30):
    x = mmlu[i]
    add("factual", "easy",
        mc_prompt(x["question"], "ABCD", x["choices"]),
        "ABCD"[x["answer"]], "multiple_choice")

print("Factual added:", counters["factual"])

README.md:   0%|          | 0.00/53.2k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/138k [00:00<?, ?B/s]

all/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.50MB            

all/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

all/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  408kB            

all/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

... (5 outputs left out)

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

Factual added: 30


In [ ]:
for cfg, diff in [("ARC-Easy", "easy"), ("ARC-Challenge", "medium")]:
    arc = load_dataset("allenai/ai2_arc", cfg, split="test")
    for i in random.sample(range(len(arc)), 15):
        x = arc[i]
        add("reasoning", diff,
            mc_prompt(x["question"], x["choices"]["label"], x["choices"]["text"]),
            x["answerKey"], "multiple_choice")

print("Reasoning added:", counters["reasoning"])

README.md:   0%|          | 0.00/9.00k [00:00<?, ?B/s]

ARC-Easy/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  331kB            

ARC-Easy/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

ARC-Easy/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  346kB            

ARC-Easy/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

ARC-Easy/validation-00000-of-00001.parqu(…): reconstructing file:   0%|          |  0.00B / 86.1kB            

... (10 outputs left out)

Generating train split:   0%|          | 0/1119 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1172 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/299 [00:00<?, ? examples/s]

Reasoning added: 30


In [ ]:
he = load_dataset("openai/openai_humaneval", split="test")
CODE_INSTR = ("\n\nComplete this Python function. Return only the full "
              "function inside a single ```python code block.")

for i in random.sample(range(len(he)), 30):
    x = he[i]
    add("code", "medium", x["prompt"] + CODE_INSTR,
        x["canonical_solution"], "code_tests",
        extra=json.dumps({"test": x["test"], "entry_point": x["entry_point"]}))

print("Code added:", counters["code"])

README.md:   0%|          | 0.00/6.52k [00:00<?, ?B/s]

openai_humaneval/test-00000-of-00001.par(…): reconstructing file:   0%|          |  0.00B / 83.9kB            

openai_humaneval/test-00000-of-00001.par(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/164 [00:00<?, ? examples/s]

Code added: 30


In [ ]:
writing = {
    "easy": [
        "Rewrite this sentence to sound more professional: 'hey, cant make it to the meeting, something came up.'",
        "Fix the grammar: 'Me and him goes to the market every sundays.'",
        "Summarize in one sentence: Regular exercise improves heart health, helps control weight, boosts mood, and improves sleep quality.",
        "Translate to Hindi: 'The library closes at 8 pm on weekdays.'",
        "Write a two-line thank-you message to a colleague who helped me finish a report.",
        "Convert this into a bulleted list: eggs, milk, bread, rice, onions.",
        "Give three synonyms for the word 'important'.",
        "Make this sentence shorter without losing meaning: 'At this point in time, we are not in a position to make a final decision.'",
        "Write a subject line for an email asking for a project deadline extension.",
        "Rewrite in simple English for a 10-year-old: 'Photosynthesis converts light energy into chemical energy.'",
    ],
    "medium": [
        "Write a 150-word email to a professor requesting a meeting to discuss a term paper topic.",
        "Write a LinkedIn post (about 120 words) announcing that I completed a data analytics course.",
        "Write a short product description (about 100 words) for a reusable steel water bottle aimed at college students.",
        "Explain the difference between revenue and profit in under 120 words with one example.",
        "Write a polite message declining a job offer while keeping the door open for the future.",
        "Write a 100-word summary of the pros and cons of remote work.",
        "Draft five interview questions for hiring a junior data analyst.",
        "Write a 150-word cover letter opening paragraph for a business analyst internship.",
        "Explain what an API is to a non-technical manager in under 100 words.",
        "Write a short apology email to a client for a delayed delivery, offering a solution.",
    ],
    "hard": [
        "Write a 300-word persuasive argument for why mid-size Indian companies should adopt AI for customer support, addressing two likely objections.",
        "Compare monopolistic competition and oligopoly in about 250 words, using one real-world industry example for each.",
        "Write a 250-word executive summary for a proposal to reduce employee attrition at a 500-person IT services firm.",
        "Explain the trade-offs between precision and recall to a business audience in about 250 words, with a fraud detection example.",
        "Write a 300-word case study opening describing a retail chain facing declining footfall, ending with three open questions for students.",
        "Write a structured 250-word critique of using GDP as the main measure of a country's well-being.",
        "Draft a 250-word go-to-market plan outline for a budget fitness app targeting Indian college students.",
        "Write a 300-word explanation of how a cost-aware LLM router saves money, including one numerical example.",
        "Write a balanced 250-word analysis of whether startups should raise venture capital or bootstrap.",
        "Write a 300-word memo to a CEO recommending whether to build or buy a CRM system, with clear reasoning.",
    ],
}

for diff, prompts in writing.items():
    for p in prompts:
        add("writing", diff, p, "", "judge")

print("Writing added:", counters["writing"])

Writing added: 30


In [ ]:
search_prompts = [
    "What are the top three news headlines in India today?",
    "What is the current repo rate set by the Reserve Bank of India?",
    "Who won the most recent Formula 1 Grand Prix?",
    "What is today's weather forecast for Bengaluru?",
    "What is the current price of gold per 10 grams in India?",
    "What was the result of the most recent Indian cricket team international match?",
    "What is the latest AI model released by OpenAI?",
    "What is the current USD to INR exchange rate?",
    "What are the latest major announcements from Google this month?",
    "What is the closing value of the Sensex on the most recent trading day?",
    "Who is the current CEO of Infosys?",
    "What new smartphones launched in India this month?",
    "What is the latest version of Python released?",
    "What are the most recent changes to India's income tax rules?",
    "What major tech company layoffs were announced this month?",
]

for p in search_prompts:
    add("search", "easy", p, "", "search_judge", needs_search=True)

print("Search added:", counters["search"])

Search added: 15


In [ ]:
df = pd.DataFrame(rows)
print("Total prompts:", len(df))
print(df.groupby(["category", "difficulty"]).size())

df.to_csv("eval_set.csv", index=False)

from google.colab import files
files.download("eval_set.csv")

Total prompts: 165
category   difficulty
code       medium        30
factual    easy          30
math       easy          10
           hard          10
           medium        10
reasoning  easy          15
           medium        15
search     easy          15
writing    easy          10
           hard          10
           medium        10
dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>